# Batch Visualize Robot Retarget Variants

Export a synchronized interactive Viser playback folder for `robot_retarget*.py` variant outputs. The folder contains `index.html`, `scene.viser`, and a local server helper for reliable browser loading.


In [ ]:
from pathlib import Path
import subprocess

ROOT = Path("/home/cipher/Codes/Academic-Projects/Retarget-whole")
HOLOSOMA = ROOT / "holosoma_retargeting" / "holosoma_retargeting"
RESULTS_ROOT = HOLOSOMA / "demo_results_total"
COMPARE_SCRIPT = HOLOSOMA / "compare_variants_viser.py"

PYTHON = ["conda", "run", "--live-stream", "-n", "robot", "python"]


## Settings

In [ ]:
TASK_TYPE = "object_interaction"
ROBOT = "g1"
TASK_NAME = "sub3_largebox_003"

LAYOUT = "grid"  # grid or overlay
SPACING = 3.0
COLUMNS = 4
FPS = 30
VISUAL_FPS_MULTIPLIER = 2
LOOP = True
LAUNCH_VIEWER = False
EXPORT_DARK_MODE = False

ROBOT_URDF = HOLOSOMA / "models" / ROBOT / f"{ROBOT}_29dof.urdf"
OBJECT_URDF = HOLOSOMA / "models" / "largebox" / "largebox.urdf"
VIEWER_OUTPUT_DIR = RESULTS_ROOT / "visualizations" / f"{TASK_NAME}_{ROBOT}_{LAYOUT}_viser_viewer"


## Variants

In [ ]:
SCRIPTS = [
    "robot_retarget.py",
    "robot_retarget_etasp.py",
    "robot_retarget_first_order.py",
    "robot_retarget_first_order_etasp.py",
    "robot_retarget_first_order_no_trust_region.py",
    "robot_retarget_first_order_no_trust_region_etasp.py",
    "robot_retarget_laplacian_smooth.py",
    "robot_retarget_laplacian_smooth_etasp.py",
    "robot_retarget_laplacian_smooth_first_order.py",
    "robot_retarget_laplacian_smooth_first_order_etasp.py",
    "robot_retarget_laplacian_smooth_first_order_no_trust_region.py",
    "robot_retarget_laplacian_smooth_first_order_no_trust_region_etasp.py",
    "robot_retarget_laplacian_smooth_no_trust_region.py",
    "robot_retarget_laplacian_smooth_no_trust_region_etasp.py",
    "robot_retarget_no_trust_region.py",
    "robot_retarget_no_trust_region_etasp.py",
]

def variant_name(script_name):
    stem = Path(script_name).stem
    if stem == "robot_retarget":
        return "original"
    return stem.removeprefix("robot_retarget_")

def result_dir_for_variant(variant):
    if TASK_TYPE in {"robot_only", "object_interaction"}:
        data_folder = "omomo"
    elif TASK_TYPE == "climbing":
        data_folder = "mocap_climb"
    else:
        raise ValueError(f"Unsupported TASK_TYPE: {TASK_TYPE}")
    return RESULTS_ROOT / variant / ROBOT / TASK_TYPE / data_folder

def expected_result_name():
    if TASK_TYPE in {"object_interaction", "climbing"}:
        return f"{TASK_NAME}_original.npz"
    if TASK_TYPE == "robot_only":
        return f"{TASK_NAME}.npz"
    raise ValueError(f"Unsupported TASK_TYPE: {TASK_TYPE}")

RESULT_NAME = expected_result_name()
VARIANTS = []
missing = []
for script_name in SCRIPTS:
    variant = variant_name(script_name)
    qpos_npz = result_dir_for_variant(variant) / RESULT_NAME
    if qpos_npz.exists():
        VARIANTS.append((variant, qpos_npz))
    else:
        missing.append((variant, qpos_npz))

print(f"Found {len(VARIANTS)} result files")
for variant, qpos_npz in VARIANTS:
    print(f"  OK     {variant:50s} {qpos_npz}")

print(f"Missing {len(missing)} result files")
for variant, qpos_npz in missing:
    print(f"  MISS   {variant:50s} {qpos_npz}")


## Export Interactive Viewer Folder

In [ ]:
if not VARIANTS:
    raise RuntimeError("No result files found to export.")

export_cmd = [
    *PYTHON,
    str(COMPARE_SCRIPT),
    "--robot-urdf", str(ROBOT_URDF),
    "--object-urdf", str(OBJECT_URDF),
    "--layout", LAYOUT,
    "--spacing", str(SPACING),
    "--columns", str(COLUMNS),
    "--fps", str(FPS),
    "--visual-fps-multiplier", str(VISUAL_FPS_MULTIPLIER),
    "--assume-object-in-qpos",
    "--export-viewer-dir", str(VIEWER_OUTPUT_DIR),
]
if LOOP:
    export_cmd.append("--loop")
else:
    export_cmd.append("--no-loop")
if EXPORT_DARK_MODE:
    export_cmd.append("--export-dark-mode")

export_cmd.append("--qpos-npzs")
export_cmd.extend(str(qpos_npz) for _variant, qpos_npz in VARIANTS)
export_cmd.append("--labels")
export_cmd.extend(variant for variant, _qpos_npz in VARIANTS)

print(" ".join(export_cmd))
subprocess.run(export_cmd, cwd=HOLOSOMA, text=True, check=True)
print(f"Interactive viewer folder: {VIEWER_OUTPUT_DIR}")
print(f"Open with: python3 {VIEWER_OUTPUT_DIR / 'serve_viewer.py'}")


## Optional Live Viewer

In [ ]:
if not LAUNCH_VIEWER:
    print("Live Viser viewer skipped. Set LAUNCH_VIEWER = True to launch it.")
else:
    if not VARIANTS:
        raise RuntimeError("No result files found to visualize.")

    cmd = [
        *PYTHON,
        str(COMPARE_SCRIPT),
        "--robot-urdf", str(ROBOT_URDF),
        "--object-urdf", str(OBJECT_URDF),
        "--layout", LAYOUT,
        "--spacing", str(SPACING),
        "--columns", str(COLUMNS),
        "--fps", str(FPS),
        "--visual-fps-multiplier", str(VISUAL_FPS_MULTIPLIER),
        "--assume-object-in-qpos",
    ]
    if LOOP:
        cmd.append("--loop")
    else:
        cmd.append("--no-loop")

    cmd.append("--qpos-npzs")
    cmd.extend(str(qpos_npz) for _variant, qpos_npz in VARIANTS)
    cmd.append("--labels")
    cmd.extend(variant for variant, _qpos_npz in VARIANTS)

    print(" ".join(cmd))
    subprocess.run(cmd, cwd=HOLOSOMA, text=True, check=True)
